# Quickstart

This notebook checks imports, runs a dummy MSFv3 forward pass, and exercises masked loss/metric functions. It does not require real SST data and does not run training.

## Data placeholders

Training scripts expect paths similar to:

```text
./data/oisst_ecs_2533_122130.zarr
./data/oisst_spatial_mask_ecs.npy
```

These files are intentionally not included in the repository.

In [ ]:
import torch

from model.models.msf3_model import MSFv3
from losses import masked_mae, masked_mse, masked_rmse

In [ ]:
torch.manual_seed(42)

model = MSFv3(
    image_size=32,
    in_chans=1,
    tin=14,
    tout=7,
    d_model=32,
    depth=1,
    num_heads=4,
    spatial_scales=(2, 4, 8, 16),
)

x = torch.randn(2, 14, 1, 32, 32)
with torch.no_grad():
    pred = model(x)

assert pred.shape == (2, 7, 1, 32, 32)
pred.shape

In [ ]:
target = torch.zeros_like(pred)
mask = torch.ones(32, 32, dtype=torch.bool)

mse = masked_mse(pred, target, mask)
rmse = masked_rmse(pred, target, mask)
mae = masked_mae(pred, target, mask)

assert torch.isfinite(mse)
assert torch.isfinite(rmse)
assert torch.isfinite(mae)
float(mse), float(rmse), float(mae)

## Training command examples

After preparing data locally and reviewing configs.py, examples are:

```bash
python model_train/base/msf3_train.py
python model_train/base/convlstm_train.py
```

These commands are examples only and should not be run as part of a lightweight smoke test.
